In [ ]:
# Day 15 - Chatbot with Conversation Memory
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

import torch
from transformers import pipeline

In [ ]:
# 1. Load LLM
pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=0 if torch.cuda.is_available() else -1,
    max_new_tokens=300,
    temperature=0.7
)

llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
# 2. Simple Memory Chatbot 
memory = ConversationBufferMemory()

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

print("Chatbot with Memory Ready!")

In [ ]:
# Test Multi-turn Conversation
print(conversation.predict(input="Hi, my name is Hayat."))
print(conversation.predict(input="What is my name?"))
print(conversation.predict(input="Tell me something about Ethiopia."))
print(conversation.predict(input="What did I ask you earlier?"))

In [ ]:
# 3. RAG + Memory (Advanced)
from langchain.chains import ConversationalRetrievalChain

# Reuse vectorstore from previous day (or recreate)
# For now, simple version:
print("\n=== RAG + Memory Chatbot ===")

def chat_with_memory(query, memory):
 
    response = conversation.predict(input=query)
    return response

# Test
print(chat_with_memory("Who are you?", memory))
print(chat_with_memory("Can you recommend Ethiopian food?", memory))